<a href="https://colab.research.google.com/github/umang0015/Meeting-Intelligence-System/blob/main/Meeting_Intelligence_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Meeting Intelligence System**



1. Project Setup
2. Import Libraries
3. Gemini API Configuration
4. Load Meeting Transcript
5. Explore Transcript
6. Basic Text Preprocessing
7. Gemini Meeting Analysis
8. Structured Output
9. Action Item Extraction
10. Decision Extraction
11. Evaluation

In [ ]:
import os
import json
import pandas as pd

from google import genai
print("Libraries imported ")

Libraries imported 


In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [ ]:
interaction = client.interactions.create(
    model="gemini-3.6-flash",
    input="Explain NLP in one sentence."
)

print(interaction.output_text)

**Natural Language Processing (NLP)** is a branch of artificial intelligence that enables computers to understand, interpret, and generate human language in both text and speech.


In [ ]:
meeting_transcript = """
Alice: Good morning everyone. Today we need to finalize the backend
architecture for our application.

Bob: I think we should use PostgreSQL as our primary database because
we need strong relational support.

Charlie: I agree with PostgreSQL. What about the API framework?

Alice: Let's use FastAPI. It will be easier for us to build the REST APIs.

Bob: Sounds good. I can prepare the database schema.

Alice: Great. Bob, please finish the database schema by Thursday.

Bob: Sure, I'll have it ready by Thursday.

Charlie: I'll work on the API implementation.

Alice: Can you finish it by Friday?

Charlie: Yes, I'll complete it by Friday.

Alice: Perfect. So we've decided to use PostgreSQL and FastAPI.
Bob will prepare the database schema by Thursday, and Charlie will
implement the API by Friday.
"""

print(meeting_transcript)


Alice: Good morning everyone. Today we need to finalize the backend
architecture for our application.

Bob: I think we should use PostgreSQL as our primary database because
we need strong relational support.

Charlie: I agree with PostgreSQL. What about the API framework?

Alice: Let's use FastAPI. It will be easier for us to build the REST APIs.

Bob: Sounds good. I can prepare the database schema.

Alice: Great. Bob, please finish the database schema by Thursday.

Bob: Sure, I'll have it ready by Thursday.

Charlie: I'll work on the API implementation.

Alice: Can you finish it by Friday?

Charlie: Yes, I'll complete it by Friday.

Alice: Perfect. So we've decided to use PostgreSQL and FastAPI.
Bob will prepare the database schema by Thursday, and Charlie will
implement the API by Friday.



In [ ]:
# number of characters
print("characters:" , len(meeting_transcript))

characters: 803


In [ ]:
# number of words
print("words:" , len(meeting_transcript.split()))

words: 131


In [ ]:
# number of lines
print("lines", len(meeting_transcript.strip().splitlines()))

lines 25


- Extract Speakers


In [ ]:
import re
speakers = re.findall(r"^([A-Za-z]+):" , meeting_transcript , re.MULTILINE)

print("speakers: ", speakers)

speakers:  ['Alice', 'Bob', 'Charlie', 'Alice', 'Bob', 'Alice', 'Bob', 'Charlie', 'Alice', 'Charlie', 'Alice']


In [ ]:
unique_speakers= list(dict.fromkeys(speakers))
print('Unique Speakers: ' , unique_speakers)

Unique Speakers:  ['Alice', 'Bob', 'Charlie']


In [ ]:
# next step is separate speaker and their speech
lines = meeting_transcript.splitlines()
conversation = []
for line in lines:
  match = re.match(r"^([A-Za-z]+):\s*(.*)" , line)

  if  match:
    speaker = match.group(1)
    text = match.group(2)

    conversation.append({
        "speaker": speaker,
        "text":text
    })


print(conversation[:3])

[{'speaker': 'Alice', 'text': 'Good morning everyone. Today we need to finalize the backend'}, {'speaker': 'Bob', 'text': 'I think we should use PostgreSQL as our primary database because'}, {'speaker': 'Charlie', 'text': 'I agree with PostgreSQL. What about the API framework?'}]


In [ ]:
# now convert it into the dataframe
import pandas as pd
conversation_df = pd.DataFrame(conversation)
conversation_df

,speaker,text
0,Alice,Good morning everyone. Today we need to finali...
1,Bob,I think we should use PostgreSQL as our primar...
2,Charlie,I agree with PostgreSQL. What about the API fr...
3,Alice,Let's use FastAPI. It will be easier for us to...
4,Bob,Sounds good. I can prepare the database schema.
5,Alice,"Great. Bob, please finish the database schema ..."
6,Bob,"Sure, I'll have it ready by Thursday."
7,Charlie,I'll work on the API implementation.
8,Alice,Can you finish it by Friday?
9,Charlie,"Yes, I'll complete it by Friday."


In [ ]:
conversation_df["speaker"].value_counts()

,count
speaker,
Alice,5
Bob,3
Charlie,3


In [ ]:
# counting how much each person spoke
speaker_counts = conversation_df["speaker"].value_counts()
speaker_counts

,count
speaker,
Alice,5
Bob,3
Charlie,3


In [ ]:
# now calculating words spoken
conversation_df["word_count"] = conversation_df["text"].apply(lambda x : len(x.split()))
conversation_df

,speaker,text,word_count
0,Alice,Good morning everyone. Today we need to finali...,10
1,Bob,I think we should use PostgreSQL as our primar...,11
2,Charlie,I agree with PostgreSQL. What about the API fr...,9
3,Alice,Let's use FastAPI. It will be easier for us to...,14
4,Bob,Sounds good. I can prepare the database schema.,8
5,Alice,"Great. Bob, please finish the database schema ...",9
6,Bob,"Sure, I'll have it ready by Thursday.",7
7,Charlie,I'll work on the API implementation.,6
8,Alice,Can you finish it by Friday?,6
9,Charlie,"Yes, I'll complete it by Friday.",6


In [ ]:
words_by_speaker = (
    conversation_df.groupby("speaker")["word_count"]
    .sum()
    .sort_values(ascending=False)


)
print(words_by_speaker)

speaker
Alice      48
Bob        26
Charlie    21
Name: word_count, dtype: int64


#### Sentence Segmentation


In [ ]:
# load spacy
import spacy
nlp = spacy.load("en_core_web_sm")
print("spacy loaded successfully")

spacy loaded successfully


In [ ]:
# test Sentence Segmentation
text = conversation_df.iloc[0]["text"]
doc = nlp(text)
for i , sentence in enumerate(doc.sents, start=1):
  print(f"Sentence{i}:{sentence.text}")

Sentence1:Good morning everyone.
Sentence2:Today we need to finalize the backend


In [ ]:
# apply sentence segmentation to the whole meeting
sentence_data= []
for _, row in conversation_df.iterrows():
  speaker = row["speaker"]
  text = row["text"]

  doc =  nlp(text)

  for sentence in doc.sents:
    sentence_data.append({
        "speaker": speaker,
        "sentence": sentence.text.strip()
    })

sentences_df = pd.DataFrame(sentence_data)
sentences_df.head(10)


,speaker,sentence
0,Alice,Good morning everyone.
1,Alice,Today we need to finalize the backend
2,Bob,I think we should use PostgreSQL as our primar...
3,Charlie,I agree with PostgreSQL.
4,Charlie,What about the API framework?
5,Alice,Let's use FastAPI.
6,Alice,It will be easier for us to build the REST APIs.
7,Bob,Sounds good.
8,Bob,I can prepare the database schema.
9,Alice,Great.


In [ ]:
# count sentences
print("Total sentences:" , len(sentences_df))

Total sentences: 17


In [ ]:
sentences_per_speaker = (
    sentences_df["speaker"]
    .value_counts
)

print(sentences_per_speaker)

<bound method IndexOpsMixin.value_counts of 0       Alice
1       Alice
2         Bob
3     Charlie
4     Charlie
5       Alice
6       Alice
7         Bob
8         Bob
9       Alice
10      Alice
11        Bob
12    Charlie
13      Alice
14    Charlie
15      Alice
16      Alice
Name: speaker, dtype: object>


In [ ]:
sentences_df.insert(
    0 ,
    "sentence_id" ,
    range(1 , len(sentences_df) + 1)

)

sentences_df.head(10)

,sentence_id,speaker,sentence
0,1,Alice,Good morning everyone.
1,2,Alice,Today we need to finalize the backend
2,3,Bob,I think we should use PostgreSQL as our primar...
3,4,Charlie,I agree with PostgreSQL.
4,5,Charlie,What about the API framework?
5,6,Alice,Let's use FastAPI.
6,7,Alice,It will be easier for us to build the REST APIs.
7,8,Bob,Sounds good.
8,9,Bob,I can prepare the database schema.
9,10,Alice,Great.


In [ ]:
# next is we dont removing stopwords
sentences_df["clean_sentence"] = (
    sentences_df["sentence"]
    .str.strip()
    .str.replace(r"\s+" , "" , regex=True)
)

sentences_df.head()

,sentence_id,speaker,sentence,clean_sentence
0,1,Alice,Good morning everyone.,Goodmorningeveryone.
1,2,Alice,Today we need to finalize the backend,Todayweneedtofinalizethebackend
2,3,Bob,I think we should use PostgreSQL as our primar...,IthinkweshouldusePostgreSQLasourprimarydatabas...
3,4,Charlie,I agree with PostgreSQL.,IagreewithPostgreSQL.
4,5,Charlie,What about the API framework?,WhatabouttheAPIframework?


In [ ]:
# questions
sentences_df["is_question"] = (
    sentences_df['sentence'].str.endswith('?')
)

sentences_df[
    sentences_df['is_question']
]

,sentence_id,speaker,sentence,clean_sentence,is_question
4,5,Charlie,What about the API framework?,WhatabouttheAPIframework?,True
13,14,Alice,Can you finish it by Friday?,CanyoufinishitbyFriday?,True


In [ ]:
# why we are building baselines
# next step is ner with spacy
# running ner on the meeting sentences
ner_results = []

for _, row in sentences_df.iterrows():
  sentence_id = row["sentence_id"]
  speaker= row["speaker"]
  sentence = row["sentence"]

  doc = nlp(sentence)

  for ent in doc.ents:
    ner_results.append({
        "sentence_id": sentence_id,
        "speaker":speaker,
        "sentence": sentence ,
        "entity" : ent.text,
        "label" : ent.label_
    })


ner_df = pd.DataFrame(ner_results)
ner_df

,sentence_id,speaker,sentence,entity,label
0,1,Alice,Good morning everyone.,morning,TIME
1,2,Alice,Today we need to finalize the backend,Today,DATE
2,3,Bob,I think we should use PostgreSQL as our primar...,PostgreSQL,GPE
3,4,Charlie,I agree with PostgreSQL.,PostgreSQL,GPE
4,5,Charlie,What about the API framework?,API,ORG
5,11,Alice,"Bob, please finish the database schema by Thur...",Bob,PERSON
6,11,Alice,"Bob, please finish the database schema by Thur...",Thursday,DATE
7,12,Bob,"Sure, I'll have it ready by Thursday.",Thursday,DATE
8,13,Charlie,I'll work on the API implementation.,API,ORG
9,14,Alice,Can you finish it by Friday?,Friday,DATE


In [ ]:
people_df= ner_df[ner_df["label"] == "PERSON"]
people_df

,sentence_id,speaker,sentence,entity,label
5,11,Alice,"Bob, please finish the database schema by Thur...",Bob,PERSON


In [ ]:
# showing dates
dates_df = ner_df[ner_df["label"] == "DATE"]

dates_df

,sentence_id,speaker,sentence,entity,label
1,2,Alice,Today we need to finalize the backend,Today,DATE
6,11,Alice,"Bob, please finish the database schema by Thur...",Thursday,DATE
7,12,Bob,"Sure, I'll have it ready by Thursday.",Thursday,DATE
9,14,Alice,Can you finish it by Friday?,Friday,DATE
10,15,Charlie,"Yes, I'll complete it by Friday.",Friday,DATE


In [ ]:
# display all entity types
entity_counts = ner_df['label'].value_counts()

print(entity_counts)

label
DATE      5
GPE       3
ORG       2
TIME      1
PERSON    1
Name: count, dtype: int64


In [ ]:
# visualize NER
from spacy import displacy
text = meeting_transcript
doc =nlp(text)

displacy.render(doc, style= 'ent', jupyter=True)

In [ ]:
def extract_entities(text):
  doc = nlp(text)

  people= []
  dates = []

  for ent in doc.ents:
    if ent.label_ == "PERSON":
      people.append(ent.text)

    elif ent.label_ == "DATE":
      dates.append(ent.text)

  return people , dates

In [ ]:
# for testing
people, dates =extract_entities(
    "Bob, please finish the database schema by Thursday."
)

print("People:" , people)
print("Dates:" , dates)

People: ['Bob']
Dates: ['Thursday']


### action item detection

- Bob -> Person
- Thursday -> Date

In [ ]:
{
    "Person" : "Bob",
    "task" : "finish the database schema",
    "deadline" : "Thursday"
}

{'Person': 'Bob', 'task': 'finish the database schema', 'deadline': 'Thursday'}

### Create a basic rule-based action detector

In [ ]:
# simple baseline
action_keywords = [
    "please" ,
    "can you" ,
    "will" ,
    "need to" ,
    "should" ,
    "responsible for" ,
    "i'll" ,
    "i will",
    "let's"
]

In [ ]:
def is_action_item(sentence):
  sentence_lower= sentence.lower()

  for keyword in action_keywords:
    if keyword in sentence_lower:
      return True

  return False

In [ ]:
test_sentences = [
    "Bob, please finish the database schema by Thursday" ,
    "What about the API framework?",
    "I think PostgreSQL is a good choice. " ,
    "Charlie will implement the API"
]

for sentence in test_sentences:
  print(sentence)
  print("Action item :" , is_action_item(sentence))
  print()

Bob, please finish the database schema by Thursday
Action item : True

What about the API framework?
Action item : False

I think PostgreSQL is a good choice. 
Action item : False

Charlie will implement the API
Action item : True



In [ ]:
# apply it to whole meeting
sentences_df["is_action_item"] = sentences_df["sentence"].apply(
    is_action_item
)

action_candidates = sentences_df[
    sentences_df["is_action_item"] == True
]

action_candidates[
    ["sentence" , "speaker" , "sentence"]
]

,sentence,speaker,sentence
1,Today we need to finalize the backend,Alice,Today we need to finalize the backend
2,I think we should use PostgreSQL as our primar...,Bob,I think we should use PostgreSQL as our primar...
5,Let's use FastAPI.,Alice,Let's use FastAPI.
6,It will be easier for us to build the REST APIs.,Alice,It will be easier for us to build the REST APIs.
10,"Bob, please finish the database schema by Thur...",Alice,"Bob, please finish the database schema by Thur..."
11,"Sure, I'll have it ready by Thursday.",Bob,"Sure, I'll have it ready by Thursday."
12,I'll work on the API implementation.,Charlie,I'll work on the API implementation.
13,Can you finish it by Friday?,Alice,Can you finish it by Friday?
14,"Yes, I'll complete it by Friday.",Charlie,"Yes, I'll complete it by Friday."


In [ ]:
# extract action item candidates
action_items = []

for _, row in action_candidates.iterrows():
  sentence = row['sentence']

  people, dates = extract_entities(sentence)

  action_items.append({
      "sentence_id" : row["sentence_id"] ,
      "speaker" : row["speaker"] ,
      "sentence" : sentence ,
      "people" : people,
      "dates" : dates
  })

# converting into dataframe

action_items_df = pd.DataFrame(action_items)

action_items_df

,sentence_id,speaker,sentence,people,dates
0,2,Alice,Today we need to finalize the backend,[],[Today]
1,3,Bob,I think we should use PostgreSQL as our primar...,[],[]
2,6,Alice,Let's use FastAPI.,[],[]
3,7,Alice,It will be easier for us to build the REST APIs.,[],[]
4,11,Alice,"Bob, please finish the database schema by Thur...",[Bob],[Thursday]
5,12,Bob,"Sure, I'll have it ready by Thursday.",[],[Thursday]
6,13,Charlie,I'll work on the API implementation.,[],[]
7,14,Alice,Can you finish it by Friday?,[],[Friday]
8,15,Charlie,"Yes, I'll complete it by Friday.",[],[Friday]


In [ ]:
# improve action item extraction
action_candidates[["sentence_id" , "speaker" , "sentence"]]

,sentence_id,speaker,sentence
1,2,Alice,Today we need to finalize the backend
2,3,Bob,I think we should use PostgreSQL as our primar...
5,6,Alice,Let's use FastAPI.
6,7,Alice,It will be easier for us to build the REST APIs.
10,11,Alice,"Bob, please finish the database schema by Thur..."
11,12,Bob,"Sure, I'll have it ready by Thursday."
12,13,Charlie,I'll work on the API implementation.
13,14,Alice,Can you finish it by Friday?
14,15,Charlie,"Yes, I'll complete it by Friday."


In [ ]:
# create context - aware person extraction
def get_responsible_person(sentence , speaker):
  people , dates =extract_entities(sentence)

  if people:
    return people[0]


  # if no explicit person is mentioned ,
  # assume the speaker is responsible

  return speaker

In [ ]:
test_sentences = "I'll work on the API implementation."

print(get_responsible_person(test_sentences , "Charlie"))

Charlie


In [ ]:
# now the next step is extract deadline
def get_deadline(sentence):
  _, dates = extract_entities(sentence)

  if dates:
    return dates[0]

  return None


In [ ]:
sentence = "Bob , please finish the database schema by Thursday."

print(get_deadline(sentence))

Thursday


In [ ]:
def extract_task(sentence):
  task= sentence

  # remove common action phrases
  patterns = [
      r"^.*?\s*please\s+",
      r"^can you\s+",
      r"^could you\s+",
      r"^i'll\s+" ,
      r"^i will\s+" ,
      r"^i'm going to\s+",
      r"^.*?\s+will\s+"
  ]

  for pattern in patterns:
    task = re.sub(pattern , "" , task , flags =re.IGNORECASE)

    # remove deadlines

    task = re.sub(
        r"\s+(by|before|until)\s+.+$" ,
        "" ,
        task ,
        flags = re.IGNORECASE
    )

    return task.strip(" .")


In [ ]:
# now the test case for extract task
test_cases = [
    "Bob, please finish the database schema by Thursday." ,
    "I'll work on the API implementation " ,
    "Charlie will implement the API by Friday."
]

for sentence in test_cases:
  print("Original:" ,  sentence)
  print("Task: " , extract_task(sentence))

  print()

Original: Bob, please finish the database schema by Thursday.
Task:  finish the database schema

Original: I'll work on the API implementation 
Task:  I'll work on the API implementation

Original: Charlie will implement the API by Friday.
Task:  Charlie will implement the API



In [ ]:
# now combining person + task + deadlines
structured_action_items= []
for _, row in action_items_df.iterrows():
  sentence_id = row["sentence_id"]
  speaker = row["speaker"]

  person = get_responsible_person(sentence , speaker)

  task = extract_task(row["sentence"])

  deadline = get_deadline(row["sentence"])

  structured_action_items.append({
      "sentence_id": row["sentence_id"] ,
      "person" : person ,
      "task" : task,
      "deadline" : deadline,
      "original_sentence" : sentence
  })

structured_action_items_df = pd.DataFrame(structured_action_items)
structured_action_items_df

,sentence_id,person,task,deadline,original_sentence
0,2,Charlie,Today we need to finalize the backend,Today,Charlie will implement the API by Friday.
1,3,Charlie,I think we should use PostgreSQL as our primar...,None,Charlie will implement the API by Friday.
2,6,Charlie,Let's use FastAPI,None,Charlie will implement the API by Friday.
3,7,Charlie,It will be easier for us to build the REST APIs,None,Charlie will implement the API by Friday.
4,11,Charlie,finish the database schema,Thursday,Charlie will implement the API by Friday.
5,12,Charlie,"Sure, I'll have it ready",Thursday,Charlie will implement the API by Friday.
6,13,Charlie,I'll work on the API implementation,None,Charlie will implement the API by Friday.
7,14,Charlie,Can you finish it,Friday,Charlie will implement the API by Friday.
8,15,Charlie,"Yes, I'll complete it",Friday,Charlie will implement the API by Friday.


In [ ]:
# build decision detection
# so starting with creating decision keywords
decision_keywords = [
    "decided" ,
    "decide" ,
    "agreed" ,
    "agree" ,
    "let's use" ,
    "we will use" ,
    "we'll use" ,
    "go with" ,
    "selected" ,
    "choose" ,
    "choosen" ,
    "finalized" ,

]

In [ ]:
# rule based decision detector
def is_decision(sentence):
  sentence_lower = sentence.lower()
  for keyword in decision_keywords:
    if keyword in sentence_lower:
      return True

  return False


In [ ]:
test_sentences = [
    "Let's use FASTAPI",
    "I agree with PostgreSQL" ,
    "What about the API framework?" ,
    "I think PostgreSQL is a good choice."
]

for sentence in test_sentences:
  print(sentence)
  print("Decision:" , is_decision(sentence))
  print()

Let's use FASTAPI
Decision: True

I agree with PostgreSQL
Decision: True

What about the API framework?
Decision: False

I think PostgreSQL is a good choice.
Decision: False



In [ ]:
# find decision sentences
sentences_df["is_decision"] = sentences_df["sentence"].apply(
    is_decision
)

decision_candidates = sentences_df[
    sentences_df["is_decision"]
]

decision_candidates[
    ["sentence_id" , "speaker" , "sentence"]
]

,sentence_id,speaker,sentence
3,4,Charlie,I agree with PostgreSQL.
5,6,Alice,Let's use FastAPI.
16,17,Alice,So we've decided to use PostgreSQL and FastAPI.


In [ ]:
# creating a basic meeting report
meeting_report  = {
    "participants": unique_speakers,
    "action_items" : structured_action_items,
    "decision" :[
        {
            "speaker" : row["speaker"] ,
            "sentence": row["sentence"]

        }

        for _, row in decision_candidates.iterrows()
    ]
}

meeting_report

{'participants': ['Alice', 'Bob', 'Charlie'],
 'action_items': [{'sentence_id': 2,
   'person': 'Charlie',
   'task': 'Today we need to finalize the backend',
   'deadline': 'Today',
   'original_sentence': 'Charlie will implement the API by Friday.'},
  {'sentence_id': 3,
   'person': 'Charlie',
   'task': 'I think we should use PostgreSQL as our primary database because',
   'deadline': None,
   'original_sentence': 'Charlie will implement the API by Friday.'},
  {'sentence_id': 6,
   'person': 'Charlie',
   'task': "Let's use FastAPI",
   'deadline': None,
   'original_sentence': 'Charlie will implement the API by Friday.'},
  {'sentence_id': 7,
   'person': 'Charlie',
   'task': 'It will be easier for us to build the REST APIs',
   'deadline': None,
   'original_sentence': 'Charlie will implement the API by Friday.'},
  {'sentence_id': 11,
   'person': 'Charlie',
   'task': 'finish the database schema',
   'deadline': 'Thursday',
   'original_sentence': 'Charlie will implement the 

In [ ]:
import json
print(json.dumps(meeting_report, indent =4))

{
    "participants": [
        "Alice",
        "Bob",
        "Charlie"
    ],
    "action_items": [
        {
            "sentence_id": 2,
            "person": "Charlie",
            "task": "Today we need to finalize the backend",
            "deadline": "Today",
            "original_sentence": "Charlie will implement the API by Friday."
        },
        {
            "sentence_id": 3,
            "person": "Charlie",
            "task": "I think we should use PostgreSQL as our primary database because",
            "deadline": null,
            "original_sentence": "Charlie will implement the API by Friday."
        },
        {
            "sentence_id": 6,
            "person": "Charlie",
            "task": "Let's use FastAPI",
            "deadline": null,
            "original_sentence": "Charlie will implement the API by Friday."
        },
        {
            "sentence_id": 7,
            "person": "Charlie",
            "task": "It will be easier for us to build the

In [ ]:
# create the gemini schema
from pydantic import BaseModel , Field
from typinh import List,Optional

ModuleNotFoundError: No module named 'typinh'